# 📊 Notebook 01: Tiền xử lý, Ẩn danh hóa & Khám phá Dữ liệu (EDA)
**Dự án:** Phân tích Cảm xúc Sinh viên Trường Đại học CMC  
**Nguồn dữ liệu:** Bình luận & bài đăng công khai từ cộng đồng sinh viên CMC (đã loại bài PR chính thức từ Fanpage trường và ẩn danh hóa PII trước khi lưu CSV).

In [1]:
import json
from collections import Counter
from pathlib import Path
import sys
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.anonymize import contains_pii
from src.preprocess import clean_text

DATA_DIR = ROOT / 'data'
REPORTS_DIR = ROOT / 'reports'
REPORTS_DIR.mkdir(exist_ok=True)
plt.rcParams['figure.dpi'] = 130

## 1. Kiểm tra phễu thu thập, lọc nhiễu & ẩn danh hóa PII

In [2]:
stats = json.loads((REPORTS_DIR / 'ingestion_stats.json').read_text(encoding='utf-8'))
split_stats = json.loads((REPORTS_DIR / 'split_stats.json').read_text(encoding='utf-8'))
print('--- THỐNG KÊ PHỄU DỮ LIỆU THẬT ---')
for k in ['n_json_files', 'total_raw_items_scanned', 'excluded_official_fanpage_posts', 'student_nonempty_records', 'after_length_filter', 'after_exact_text_clean_dedup', 'unique_core_text_count', 'pii_residual_count']:
    print(f'{k:35s}: {stats[k]}')
print('Train / Val / Test (dedup core):', split_stats['n_train'], '/', split_stats['n_val'], '/', split_stats['n_test'])

--- THỐNG KÊ PHỄU DỮ LIỆU THẬT ---
n_json_files                       : 3
total_raw_items_scanned            : 2763
excluded_official_fanpage_posts    : 751
student_nonempty_records           : 2002
after_length_filter                : 1936
after_exact_text_clean_dedup       : 1868
unique_core_text_count             : 1860
pii_residual_count                 : 0
Train / Val / Test (dedup core): 1302 / 279 / 279


## 2. Đọc dữ liệu đã làm sạch & gán nhãn bán tự động

In [3]:
df = pd.read_csv(DATA_DIR / 'cmc_student_labeled.csv', encoding='utf-8-sig')
df['date_dt'] = pd.to_datetime(df['date'], errors='coerce')
df['char_len'] = df['text_clean'].fillna('').str.len()
df['word_len'] = df['text_clean'].fillna('').str.split().str.len()
assert int(df['text'].fillna('').apply(contains_pii).sum()) == 0
print('Tổng mẫu sạch:', len(df))
print('Phân bố nhãn:', df['label'].value_counts().to_dict())
print('Độ dài TB (từ):', round(df['word_len'].mean(), 2), '| Trung vị:', df['word_len'].median())
df[['id', 'type', 'source', 'date', 'topic', 'label', 'text_clean']].head(5)

Tổng mẫu sạch: 1868
Phân bố nhãn: {'Neutral': 919, 'Positive': 640, 'Negative': 309}
Độ dài TB (từ): 30.64 | Trung vị: 10.0


,id,type,source,date,topic,label,text_clean
0,cmc_7edd9bb0c0ba,comment,ctsv.cmcuni,2025-11-29,Giao lưu & Khác,Positive,mê cái cách cổ edit thanh thuỷ
1,cmc_943fdaf2ac5b,comment,ctsv.cmcuni,2025-11-30,Hoạt động & CLB,Positive,câu lạc bộ the rebels chúng em rất vui khi đã được góp phần tạo nên kỉ niệm cho các anh chị k1 một kỉ niệm đáng nhớ cảm ơn trường đại học cmc đã tạo đất diễn cũng như sự cố gắng của các câu lạc bộ múa gim câu lạc bộ múa trường đại học cmc và câu lạc bộ nhảy the ix cmc dance club
2,cmc_1e569210fb62,comment,ctsv.cmcuni,2025-12-07,Giao lưu & Khác,Positive,quá mê cmc
3,cmc_71dd59f2b651,comment,ctsv.cmcuni,2025-12-12,Giao lưu & Khác,Positive,em thích anh tú
4,cmc_e5fe94513452,comment,ctsv.cmcuni,2025-12-12,Học vụ & Giảng viên,Positive,chúc mừng các cô gái cmc tôi đã theo dõi 2 trận đấu của bác bạn tôi xin góp ý một chút kinh nghiệm cá nhân các bạn khi nhận được bóng phải thật bình tĩnh sử lý đúng 3 chuyền và phải giữ tinh thần khi để thua mất 1 điểm hoặc vài điểm tinh thần chiến đấu phải đững vững sự nản lòng sẽ kèm theo thua cuộc và mất đi chiến thắng ngay trong trận đấu chúc các bạn cmc vào những trận tiếp theo hãy chiến thắng nhé các cô gái


## 3. Biểu đồ phân bố cảm xúc & nguồn/loại nội dung

In [4]:
COLORS = {'Positive': '#10B981', 'Neutral': '#3B82F6', 'Negative': '#EF4444'}
order = ['Positive', 'Neutral', 'Negative']
counts = df['label'].value_counts().reindex(order, fill_value=0)

fig, ax = plt.subplots(figsize=(7, 4.2))
bars = ax.bar(counts.index, counts.values, color=[COLORS[c] for c in counts.index], edgecolor='black')
ax.set_title('Phân bố Cảm xúc Cộng đồng Sinh viên CMC (Dữ liệu thật đã ẩn danh)', fontweight='bold')
ax.set_ylabel('Số lượng bản ghi')
for b in bars:
    h = b.get_height()
    ax.text(b.get_x() + b.get_width()/2, h, f'{int(h)} ({h/len(df)*100:.1f}%)', ha='center', va='bottom', fontweight='bold')
ax.grid(axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.savefig(REPORTS_DIR / 'eda_sentiment_dist.png', dpi=150)
plt.close(fig)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
src_cnt = df['source'].value_counts()
axes[0].barh(src_cnt.index, src_cnt.values, color='#0EA5E9', edgecolor='black')
axes[0].set_title('Phân bố theo Nguồn (bài Group + bình luận SV)', fontweight='bold')
axes[0].set_xlabel('Số lượng')
type_cnt = df['type'].value_counts()
axes[1].bar(type_cnt.index, type_cnt.values, color=['#6366F1', '#14B8A6'], edgecolor='black')
axes[1].set_title('Tỷ trọng Bài đăng (post) vs Bình luận (comment)', fontweight='bold')
axes[1].set_ylabel('Số lượng')
plt.tight_layout()
plt.savefig(REPORTS_DIR / 'eda_source_type.png', dpi=150)
plt.close(fig)
print('Đã lưu eda_sentiment_dist.png và eda_source_type.png')

Đã lưu eda_sentiment_dist.png và eda_source_type.png


## 4. Biểu đồ phân bố cảm xúc theo 6 chủ đề sinh viên

In [5]:
ct = pd.crosstab(df['topic'], df['label']).reindex(columns=order, fill_value=0)
ax = ct.plot(kind='barh', stacked=True, figsize=(10, 5), color=[COLORS[c] for c in order], edgecolor='black')
ax.set_title('Phân bố Cảm xúc theo 6 Chủ đề Thảo luận của Sinh viên CMC', fontweight='bold')
ax.set_xlabel('Số lượng bài đăng & bình luận')
ax.set_ylabel('Chủ đề')
ax.grid(axis='x', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.savefig(REPORTS_DIR / 'eda_topic_sentiment.png', dpi=150)
plt.close()
ct

label,Positive,Neutral,Negative
topic,,,
Cơ sở vật chất & CNTT,20,39,12
Giao lưu & Khác,420,645,236
Hoạt động & CLB,108,75,24
Học vụ & Giảng viên,47,68,24
Tìm đồ & Hỗ trợ,7,33,7
Đời sống & Nhà trọ,38,59,6


## 5. Phân phối độ dài câu & xu hướng cảm xúc theo tuần

In [6]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
clipped = df['word_len'].clip(upper=100)
axes[0].hist(clipped, bins=25, color='#6366F1', edgecolor='white')
axes[0].axvline(df['word_len'].mean(), color='#EF4444', linestyle='--', label=f"TB: {df['word_len'].mean():.1f} từ")
axes[0].set_title('Phân phối độ dài câu (giới hạn 100 từ)', fontweight='bold')
axes[0].set_xlabel('Số từ')
axes[0].set_ylabel('Tần suất')
axes[0].legend()

sns.boxplot(data=df[df['word_len'] <= 80], x='label', y='word_len', order=order, hue='label', palette=COLORS, legend=False, ax=axes[1])
axes[1].set_title('Độ dài câu theo Nhãn cảm xúc', fontweight='bold')
axes[1].set_xlabel('Nhãn')
axes[1].set_ylabel('Số từ')
plt.tight_layout()
plt.savefig(REPORTS_DIR / 'eda_length_dist.png', dpi=150)
plt.close(fig)

tdf = df.dropna(subset=['date_dt']).copy()
tdf['week'] = tdf['date_dt'].dt.to_period('W').apply(lambda p: p.start_time)
weekly = tdf.groupby(['week', 'label']).size().unstack(fill_value=0).reindex(columns=order, fill_value=0)
recent = weekly.tail(14)
fig, ax = plt.subplots(figsize=(11, 4.5))
for lbl in order:
    ax.plot(recent.index, recent[lbl], marker='o', label=lbl, color=COLORS[lbl], linewidth=2)
ax.set_title('Xu hướng Cảm xúc Sinh viên CMC theo Tuần (14 tuần gần nhất)', fontweight='bold')
ax.set_ylabel('Số lượng bản ghi')
ax.legend()
ax.grid(True, linestyle='--', alpha=0.4)
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig(REPORTS_DIR / 'eda_weekly_trend.png', dpi=150)
plt.close(fig)
print('Đã lưu eda_length_dist.png và eda_weekly_trend.png')

Đã lưu eda_length_dist.png và eda_weekly_trend.png


## 6. Kiểm tra chất lượng tiền xử lý & từ khóa phổ biến
Xác nhận các lỗi teencode trước đây (`cmc -> đại học đại học`, `bt -> bình thường`, `k5`) đã được xử lý sạch.

In [7]:
samples = [
    'Trường Đại học CMC tuyển sinh K5 và K4',
    'Ai bt lịch thi ck chưa, mình ko bt luôn',
    'CSVC phòng lab xịn quáaaaa, liên hệ 0912345678 hoặc sv@cmc.edu.vn'
]
for s in samples:
    print('Gốc  :', s)
    print('Sạch :', clean_text(s))
    print('-'*40)

top_words = Counter(' '.join(df['text_clean'].fillna('')).split()).most_common(15)
print('Top 15 từ phổ biến:', top_words)

Gốc  : Trường Đại học CMC tuyển sinh K5 và K4
Sạch : trường đại học cmc tuyển sinh k5 và k4
----------------------------------------
Gốc  : Ai bt lịch thi ck chưa, mình ko bt luôn
Sạch : ai bt lịch thi cuối kỳ chưa mình không bt luôn
----------------------------------------
Gốc  : CSVC phòng lab xịn quáaaaa, liên hệ 0912345678 hoặc sv@cmc.edu.vn
Sạch : cơ sở vật chất phòng lab xịn quáa liên hệ hoặc
----------------------------------------
Top 15 từ phổ biến: [('có', 865), ('bạn', 684), ('mình', 652), ('và', 629), ('không', 621), ('ạ', 530), ('học', 519), ('trường', 480), ('cho', 450), ('là', 442), ('những', 428), ('các', 424), ('được', 416), ('người', 405), ('cmc', 401)]
